# NB_lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test type per calendar month) from silver lab_results, enriched with month and quarter names from the calendar dimension, with a LAG-based month-over-month abnormal-rate trend, per the STTM `Gold/sttm_silver_to_gold.xlsx`. Load type: FULL rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
# Catalog placeholder from the STTM (<catalog>); can be overridden by ADF / Jobs
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "adb_caresync"

silver = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Source schema : {silver}")
print(f"Target table  : {target_table}")

In [0]:
# Driving table: silver.lab_results grouped by (test_name, year, month of test_date)
# LEFT JOIN calendar on date_key = yyyyMMdd(first day of month) for month_name / quarter_name
# LAG(abnormal_rate_pct) per test_name ordered by year, month -> month-over-month change (percentage points)
sql_text = f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per lab test type per month with volumes, abnormal/critical rates and month-over-month trend (source: STTM silver_to_gold)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                AS year,
    month(test_date)                               AS month,
    CAST(date_trunc('MONTH', test_date) AS DATE)   AS month_start_date,
    COUNT(*)                                       AS total_tests,
    COUNT_IF(result_status = 'Abnormal')           AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')           AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('MONTH', test_date) AS DATE)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name                                                        AS test_name,
  CAST(t.year AS INT)                                                AS year,
  CAST(t.month AS INT)                                               AS month,
  c.month_name                                                       AS month_name,
  c.quarter_name                                                     AS quarter_name,
  CAST(t.total_tests AS INT)                                         AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                 AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                 AS critical_test_count,
  t.abnormal_rate_pct                                                AS abnormal_rate_pct,
  t.critical_rate_pct                                                AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))               AS prev_month_abnormal_rate_pct,
  CASE
    WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                                                AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN {silver}.calendar c
  ON c.date_key = CAST(date_format(t.month_start_date, 'yyyyMMdd') AS INT)
"""
print(sql_text)
spark.sql(sql_text)
print(f"Created {target_table}")

In [0]:
# Return the gold record count to the caller (ADF / Jobs / dbutils.notebook.run)
updated_df = spark.table(target_table)
record_count = updated_df.count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))